In [72]:
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

In [73]:
df_raw = pd.read_csv('../data/cps_00021.csv')
exposure_scores = pd.read_csv('../data/final_exposure.csv')

df_raw = df_raw[(df_raw['OCC'] != 0) & (df_raw['OCC'].notna())]
df_raw = df_raw[(df_raw['EMPSTAT'] == 10) | (df_raw['EMPSTAT'] == 12)]
df_raw = df_raw[(df_raw['AGE'] >= 18)]

In [74]:
# CPS occupation
OCC_COL = 'OCC'
# CPS weight
WEIGHT = 'WTFINL'
# CPS age
AGE_COL = 'AGE'
# Young/Entry Workers Column
Y_COL = '18_25'
# CPS industry
IND_COL = 'IND'
# CPS weight  
WEIGHT_COL    = 'WTFINL'                  
# Scores occupation       
INDEX_OCC_COL = 'CPS Code'
# All score columns
INDEX_COLS = ['alpha', 'beta', 'gamma']

In [75]:
# Convert both OCC and CPS Code to string columns
df_raw[OCC_COL] = df_raw[OCC_COL].astype(float).astype(int).astype(str)
exposure_scores[INDEX_OCC_COL] = exposure_scores[INDEX_OCC_COL].astype(float).astype(int).astype(str)

In [76]:
# Merge exposure scores onto the dataset
df = df_raw.merge(exposure_scores,
                     left_on="OCC",
                     right_on=INDEX_OCC_COL,
                     how='left',
                     indicator=True)

# 32203 Rows do not merge
failed = df['_merge'] == 'left_only'
failed_to_merge = df[failed]

# 12 occupation codes did not merge
# They are residuals (All Other) or Military occupations (9840)
# Onet does not track them so they are being dropped
# https://www.onetcenter.org/dl_files/Taxonomy2019_Summary.pdf
# See citation 4 on page 10

print(f"Failed to merge {failed_to_merge['OCC'].nunique()} rows with exposure scores.")
assert failed_to_merge['OCC'].nunique() == 12, "Some rows did not merge or somehow not in ONET codes merged"

df = df.drop(columns=["CPS Code", "CPS Title", "_merge"])

df_scores = df[~failed]

Failed to merge 12 rows with exposure scores.


In [77]:
# Construct panels at 4 different window lengths, check for cell size filters.
# Since the actual proportions per occupation are fairly low,
# Many low N and low n cells provide unreliable point estimates of the true proportion

# Create count of young workers
df_scores[Y_COL] = df_scores[AGE_COL].between(18, 25).copy()
df_scores["TECH"] = df_scores[IND_COL].astype(str).isin(["6490", "6672", "6695", "7380"]).copy()

# Aggregate to quarter, half year levels using treatment date as time = 0
treat_year = 2022
treat_month = 11

df_scores['MONTHS_FROM_TREAT'] = (df_scores["YEAR"] - treat_year) * 12 + (df_scores['MONTH'] - treat_month)
df_scores['QUARTER'] = df_scores['MONTHS_FROM_TREAT'] // 3
df_scores['HALF_YEAR'] = df_scores['MONTHS_FROM_TREAT'] // 6

# Drop trailing months when aggregating:

# Take number of months within every period specification and trim if the number of months is less than the specified window_length
# Tolerance -- we keep 2025H2 and 2025 but not 2025Q4 because of the October CPS
def drop_thin_periods(df, period_col, window_length, tolerance=1):
    counts = df.groupby(period_col)[['YEAR','MONTH']].apply(lambda g: g.drop_duplicates().shape[0])
    keep = counts[counts >= window_length - tolerance].index
    return df[df[period_col].isin(keep)]

df_q = drop_thin_periods(df_scores, 'QUARTER', 3, tolerance=0)
df_h = drop_thin_periods(df_scores, 'HALF_YEAR', 6, tolerance=1)

In [78]:
def compute_cell_filters(df_w, window_col, n_min=30, np_min=5):
    # We only filter on pre-treatment data
    df_w = df_w[(df_w[window_col] < 0)].copy()

    df_w["_wy"] = df_w["WTFINL"] * df_w[Y_COL]
    df_w["_wtech"] = df_w["WTFINL"] * df_w["TECH"]

    cells_w = df_w.groupby(['OCC', window_col]).agg(
        N=(AGE_COL, 'size'),
        np=(Y_COL, 'sum'),
        tech=("TECH", 'sum'),
        _wN=('WTFINL', 'sum'),
        _wnp=('_wy', 'sum'),
        _wtech=('_wtech', 'sum'))
    cells_w['nq'] = cells_w['N'] - cells_w['np']

    cell_counts_N = cells_w['N'].unstack(fill_value=0)
    cell_counts_np = cells_w['np'].unstack(fill_value=0)
    cell_counts_nq = cells_w['nq'].unstack(fill_value=0)

    valid = (cell_counts_N >= n_min) & (cell_counts_np >= np_min) & (cell_counts_nq >= np_min)
    passing_occ = valid.index[valid.all(axis=1)]
    pass_counts = valid.sum(axis=0)

    return {
        "passing_occ": passing_occ,
        "pass_counts": pass_counts,
        "valid_matrix": valid,
        "cell_counts": {
            "N": cell_counts_N,
            "np": cell_counts_np,
            "nq": cell_counts_nq
        }
    }

In [79]:
windows = {
    'monthly': (df_scores, 'MONTHS_FROM_TREAT'),
    'quarterly': (df_q, 'QUARTER'),
    'half-year': (df_h, 'HALF_YEAR')
}

filter_results = {}

for name, (df_w, window_col) in windows.items():
    filter_results[name] = compute_cell_filters(df_w, window_col)
    n_passing = len(filter_results[name]['passing_occ'])
    print(f"PASSING occupations ({name}): {n_passing}, Percent: {(n_passing/513)*100}%")

PASSING occupations (monthly): 108, Percent: 21.052631578947366%
PASSING occupations (quarterly): 222, Percent: 43.27485380116959%
PASSING occupations (half-year): 311, Percent: 60.62378167641326%


In [80]:
# Testing for selection bias
# Do cell-filter failing and passing occupations differ in their exposure scores

for index in INDEX_COLS:
    print(f'FOR INDEX {index}:')
    results_exposure = []
    for name, (df_w, window_col) in windows.items():
        df_w = df_w[(df_w[window_col] < 0)]
        passing_w = filter_results[name]['passing_occ']
        t_stat, p_val = stats.ttest_ind(
            df_w[df_w['OCC'].isin(passing_w)].groupby('OCC')[index].mean(),
            df_w[~df_w['OCC'].isin(passing_w)].groupby('OCC')[index].mean(),
            equal_var=False
        )
        results_exposure.append({'window': name, 't_stat': t_stat, 'p_val': p_val})
    results_exposure_df = pd.DataFrame(results_exposure)
    print(results_exposure_df)


# No statistically different results between filtered and unfiltered groups at any level across any index

FOR INDEX alpha:
      window    t_stat     p_val
0    monthly -0.402902  0.687489
1  quarterly  0.752362  0.452203
2  half-year  1.723114  0.085547
FOR INDEX beta:
      window    t_stat     p_val
0    monthly -0.578181  0.563912
1  quarterly  1.103619  0.270320
2  half-year  1.732393  0.083899
FOR INDEX gamma:
      window    t_stat     p_val
0    monthly -0.554609  0.579912
1  quarterly  1.064364  0.287708
2  half-year  1.399149  0.162480


In [81]:
results_share = []
for name, (df_w, window_col) in windows.items():
    df_w = df_w[(df_w[window_col] < 0)]
    passing_w = filter_results[name]['passing_occ']
    
    passing_vals = df_w[df_w['OCC'].isin(passing_w)].groupby('OCC')[Y_COL].mean()
    failing_vals = df_w[~df_w['OCC'].isin(passing_w)].groupby('OCC')[Y_COL].mean()
    t_stat, p_val = stats.ttest_ind(passing_vals, failing_vals, equal_var=False)
    diff = passing_vals.mean() - failing_vals.mean()
    
    # Baseline: average of occupation-level means across all occupations
    overall_occ_mean = df_w.groupby('OCC')[Y_COL].mean().mean()
    pct_diff = (diff / overall_occ_mean) * 100
    results_share.append({
        'window': name, 
        't_stat': t_stat, 
        'p_val': p_val, 
        'diff': diff, 
        'pct_diff': pct_diff
    })

print(pd.DataFrame(results_share))

# In every window filtered out cells have less young workers
# However based on CLT for bionmial proportions these estimates may not be accurate
# Will include high N groups in robustness check even if they are low np

      window    t_stat         p_val      diff   pct_diff
0    monthly  5.115482  1.021594e-06  0.061038  54.466181
1  quarterly  6.704598  7.511268e-11  0.057100  50.952273
2  half-year  6.559444  1.342003e-10  0.051147  45.639674


In [82]:
for name, (df_w, window_col) in windows.items():
    print(df_w.columns)

Index(['YEAR', 'SERIAL', 'MONTH', 'HWTFINL', 'CPSID', 'ASECFLAG', 'PERNUM',
       'WTFINL', 'CPSIDP', 'CPSIDV', 'AGE', 'EMPSTAT', 'OCC', 'IND', 'alpha',
       'beta', 'gamma', 'alpha_z', 'beta_z', 'gamma_z', '18_25', 'TECH',
       'MONTHS_FROM_TREAT', 'QUARTER', 'HALF_YEAR'],
      dtype='str')
Index(['YEAR', 'SERIAL', 'MONTH', 'HWTFINL', 'CPSID', 'ASECFLAG', 'PERNUM',
       'WTFINL', 'CPSIDP', 'CPSIDV', 'AGE', 'EMPSTAT', 'OCC', 'IND', 'alpha',
       'beta', 'gamma', 'alpha_z', 'beta_z', 'gamma_z', '18_25', 'TECH',
       'MONTHS_FROM_TREAT', 'QUARTER', 'HALF_YEAR'],
      dtype='str')
Index(['YEAR', 'SERIAL', 'MONTH', 'HWTFINL', 'CPSID', 'ASECFLAG', 'PERNUM',
       'WTFINL', 'CPSIDP', 'CPSIDV', 'AGE', 'EMPSTAT', 'OCC', 'IND', 'alpha',
       'beta', 'gamma', 'alpha_z', 'beta_z', 'gamma_z', '18_25', 'TECH',
       'MONTHS_FROM_TREAT', 'QUARTER', 'HALF_YEAR'],
      dtype='str')


In [83]:
for name, (df_w, window_col) in windows.items():
        df_w = df_w[(df_w[window_col] < 0)]
        emp_share = df_w.groupby('OCC')[WEIGHT_COL].sum()
        passing_w = filter_results[name]['passing_occ']
        print(f"Passing occ share of total employment of sample({name}: {(emp_share[passing_w].sum() / emp_share.sum())*100}%")

Passing occ share of total employment of sample(monthly: 68.62233880345501%
Passing occ share of total employment of sample(quarterly: 85.7045048821029%
Passing occ share of total employment of sample(half-year: 93.53524040828853%


In [91]:
def convert_to_panel(df_w, window_col, passing_occs):
    if passing_occs is not None:
        df_w = df_w[df_w['OCC'].isin(passing_occs)].copy()

    # 2. Compute weighted micro-variables
    df_w["_wy"] = df_w["WTFINL"] * df_w[Y_COL]
    df_w["_wtech"] = df_w["WTFINL"] * df_w["TECH"]
    
    # Build aggregation dictionary dynamically
    agg_dict = {
        'N': (AGE_COL, 'size'),
        'np': (Y_COL, 'sum'),
        'tech': ("TECH", 'sum'),
        '_wN': ('WTFINL', 'sum'),
        '_wnp': ('_wy', 'sum'),
        '_wtech': ('_wtech', 'sum')
    }
    agg_dict.update({col: (col, 'first') for col in INDEX_COLS})

    panel_w = df_w.groupby(['OCC', window_col]).agg(**agg_dict).reset_index()

    panel_w['Y_SHARE'] = panel_w['np'] / panel_w['N']
    panel_w['TECH_SHARE'] = panel_w['tech'] / panel_w['N']
    panel_w['W_Y_SHARE'] = panel_w['_wnp'] / panel_w['_wN']
    panel_w['W_TECH_SHARE'] = panel_w['_wtech'] / panel_w['_wN']

    for col in INDEX_COLS:
        quintile_col = f"{col}_quintile"
        panel_w[quintile_col] = pd.qcut(panel_w[col], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
        dummies = pd.get_dummies(panel_w[quintile_col], prefix=quintile_col, dtype=int)
        panel_w[dummies.columns] = dummies

    return panel_w

In [94]:
panel_q = convert_to_panel(df_q, 'QUARTER', filter_results['quarterly']['passing_occ'])
panel_h = convert_to_panel(df_h, 'HALF_YEAR', filter_results['half-year']['passing_occ'])
panel_q_full = convert_to_panel(df_q, 'QUARTER', None)

In [95]:
panel_q.to_csv('../data/checkpoints/panel_q.csv', index=False)
panel_h.to_csv('../data/checkpoints/panel_h.csv', index=False)
panel_q_full.to_csv('../data/checkpoints/panel_q_full.csv', index=False)

passing_occs_df = pd.concat([
    pd.DataFrame({'window': window_name, 'OCC': list(res['passing_occ'])})
    for window_name, res in filter_results.items()
], ignore_index=True)

passing_occs_df.to_json('../data/checkpoints/passing_occs.json', orient='records', indent=2)